# 3.9 — Lasso & Sparsity

Lasso adds an L1 penalty to ordinary squared-error regression, and that one change can set coefficients exactly to zero. In this lesson, you will build the loss, the penalty, the sparsity mechanism, and the validation decision rule from scratch with NumPy so the model-selection arithmetic never becomes a black box.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Lasso one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the L1 corner and soft-threshold update, is shown directly. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + linear algebra for losses, penalties, and coordinate updates.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the tiny synthetic datasets.

### 1. Empirical risk: the average loss the learner tries to improve

Every regularized model starts with a plain training score: the empirical risk, or average loss over observed examples. In this lesson's verified toy arithmetic, the three per-example losses are 0.180, 0.135, and 0.522, so the empirical risk is

$$R_S=\frac{0.180+0.135+0.522}{3}=0.279.$$

That average is deliberately tiny enough to check by hand because larger ML systems still use the same move: add the losses you saw, divide by how many examples created them, and only then compare models.

In [ ]:
losses_w = np.array([0.180, 0.135, 0.522])  # three verified per-example losses.
print("losses:", losses_w)                 # inspect the training fragments.
print("sum:", round(float(losses_w.sum()), 3))  # 0.837 total loss over three examples.

▶ What you'll see: the raw losses and their total, before any averaging or penalty is added.

In [ ]:
risk_w = float(losses_w.mean())             # empirical risk = average training loss.
print("empirical risk R_S:", round(risk_w, 3))  # 0.279.
assert round(risk_w, 3) == 0.279            # matches the lesson arithmetic.

▶ What you'll see: the empirical risk is 0.279, the raw fit term the optimizer wants to reduce.

In [ ]:
plt.figure(figsize=(4.4, 3))                         # compact view of the three losses.
plt.bar(["ex1", "ex2", "ex3"], losses_w, color="steelblue")  # each bar is one example's contribution.
plt.axhline(risk_w, color="crimson", linestyle="--", label=f"mean={risk_w:.3f}")  # average reference.
plt.ylabel("loss"); plt.title("1: empirical risk is an average"); plt.legend(); plt.show()

▶ What you'll see: one large loss can pull the average upward, so the mean is a summary of all examples rather than the best-looking one.

*Why it's done this way: empirical risk is an unbiased accounting unit only after dividing by sample size; otherwise a model evaluated on more examples would look worse merely because it had more terms to sum.*

### 2. Adding a cost term: the selection score is not the raw fit alone

Lasso does not choose coefficients by training loss alone. It optimizes a fit term plus a cost for model complexity. In the lesson prose, that cost is 0.080, so the selection score is

$$score=R_S+cost=0.279+0.080=0.359.$$

This is the practical reason regularization matters: a tempting raw fit is allowed to win only if it still wins after paying for flexibility.

In [ ]:
cost_w = 0.080                              # regularization, complexity, or operational cost.
score_w = risk_w + cost_w                   # full decision score.
print("raw risk:", round(risk_w, 3))        # 0.279.
print("cost:", round(cost_w, 3))            # 0.080.

▶ What you'll see: the raw term and the extra cost are separate pieces of the model-selection score.

In [ ]:
print("score = risk + cost:", round(score_w, 3))  # 0.359.
assert round(score_w, 3) == 0.359                  # matches the lesson arithmetic.

▶ What you'll see: the full score is 0.359, not the prettier raw training number 0.279.

In [ ]:
plt.figure(figsize=(4.4, 3))                              # show the score decomposition.
plt.bar(["risk", "cost", "total"], [risk_w, cost_w, score_w], color=["seagreen", "orange", "purple"])
plt.ylabel("score component"); plt.title("2: pay the cost before comparing"); plt.show()

▶ What you'll see: the total bar is the only number that should be compared to another model's total score.

*Why it's done this way: the penalty changes the objective from "fit this sample" to "fit this sample with a controlled amount of complexity," which is the version more likely to survive validation.*

### 3. The Lasso objective: squared error plus an L1 penalty

For a linear model $\hat y=X\beta$, Lasso solves

$$\hat\beta=\arg\min_\beta \frac12\lVert y-X\beta\rVert_2^2+\lambda\lVert\beta\rVert_1,$$

where $\lVert\beta\rVert_1=\sum_j |\beta_j|$. The first term asks predictions to match data; the second charges one unit for every unit of absolute coefficient size. The factor $1/2$ is only for clean derivatives: differentiating $\frac12 e^2$ gives $e$ instead of $2e$.

In [ ]:
X_w = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0]])  # tiny design matrix.
y_w = np.array([1.0, 1.0, 2.0, 2.5])                              # target values.
beta_w = np.array([0.8, 0.6])                                      # one candidate coefficient vector.
print("X shape:", X_w.shape, "beta:", beta_w)                    # inspect model dimensions.

▶ What you'll see: four examples, two features, and one candidate coefficient vector.

In [ ]:
pred_w = X_w @ beta_w                         # linear predictions.
resid_w = y_w - pred_w                        # residuals y - X beta.
sse_half_w = 0.5 * float(np.sum(resid_w ** 2))  # half squared-error fit term.
l1_w = float(np.sum(np.abs(beta_w)))          # L1 norm = sum of absolute coefficients.
print("predictions:", np.round(pred_w, 3))    # inspect fit.
print("half SSE:", round(sse_half_w, 3), "L1:", round(l1_w, 3))  # inspect both objective pieces.

▶ What you'll see: the fit term and the L1 size term are computed on different objects: residuals versus coefficients.

In [ ]:
lam_w = 0.4                                   # penalty strength.
obj_w = sse_half_w + lam_w * l1_w             # Lasso objective value.
print("objective:", round(obj_w, 3))          # 0.325 + 0.4*1.4 = 0.885.
assert round(obj_w, 3) == 0.885               # concrete numeric check.

▶ What you'll see: a single candidate score that includes both fit and coefficient cost.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["0.5||y-Xβ||²", "λ||β||₁", "total"], [sse_half_w, lam_w * l1_w, obj_w], color=["steelblue", "orange", "purple"])
plt.xticks(rotation=15); plt.ylabel("objective value"); plt.title("3: Lasso objective pieces"); plt.show()

▶ What you'll see: increasing λ would enlarge only the penalty bar, changing which β is best.

*Why it's done this way: squared error rewards accurate predictions while L1 charges absolute coefficient mass, so a feature must earn its coefficient by reducing error more than it increases penalty.*

### 4. Why L1 creates exact zeros: the corner at zero

The special feature of L1 is not merely that it shrinks coefficients; it has a sharp corner at zero. For one standardized coordinate, the simplified problem is

$$\min_b \frac12(b-z)^2+\lambda |b|.$$

If $z$ is the unpenalized coefficient, the L1 solution is soft-thresholding:

$$b^*=\operatorname{sign}(z)\max(|z|-\lambda,0).$$

When $|z|\le \lambda$, the maximum is zero, so the coefficient becomes exactly zero. That is sparsity.

In [ ]:
z_grid_w = np.linspace(-2.0, 2.0, 401)              # possible unpenalized coordinate values.
lam_soft_w = 0.7                                    # threshold width around zero.
soft_w = np.sign(z_grid_w) * np.maximum(np.abs(z_grid_w) - lam_soft_w, 0.0)  # L1 solution.
print("soft(-0.5), soft(0.5), soft(1.2):", np.round(np.interp([-0.5, 0.5, 1.2], z_grid_w, soft_w), 3))

▶ What you'll see: values whose magnitude is below 0.7 map to exactly 0, while 1.2 shrinks to about 0.5.

In [ ]:
z_test_w = np.array([-1.4, -0.4, 0.0, 0.4, 1.4])  # five easy-to-check z values.
soft_test_w = np.sign(z_test_w) * np.maximum(np.abs(z_test_w) - lam_soft_w, 0.0)
print("z:", z_test_w)                             # unpenalized coordinate values.
print("soft-thresholded:", soft_test_w)           # Lasso coordinate optima.
assert np.allclose(soft_test_w, [-0.7, 0.0, 0.0, 0.0, 0.7])  # exact sparsity check.

▶ What you'll see: three middle values become exactly zero, not merely small.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.plot(z_grid_w, soft_w, color="purple", label="L1 soft-threshold")
plt.plot(z_grid_w, z_grid_w, color="gray", linestyle="--", label="no penalty")
plt.axvspan(-lam_soft_w, lam_soft_w, color="orange", alpha=0.18, label="zero region")
plt.xlabel("unpenalized z"); plt.ylabel("Lasso coefficient b*"); plt.title("4: L1 makes an exact-zero zone"); plt.legend(); plt.show()

▶ What you'll see: a flat segment at zero; that flat segment is the algebraic source of sparse models.

*Why it's done this way: the absolute-value penalty has a subgradient interval at zero, so small data gradients can be balanced without moving away from zero; L2 lacks that corner and usually only shrinks coefficients continuously.*

### 5. Coordinate descent: solving Lasso by one coefficient at a time

A full Lasso fit can be built by repeating the one-coordinate soft-threshold update. If columns are standardized, each coordinate sees a partial residual and computes

$$\rho_j=x_j^\top(y-X_{-j}\beta_{-j}),\qquad
\beta_j\leftarrow \frac{S(\rho_j,\lambda)}{x_j^\top x_j},$$

where $S(\rho,\lambda)=\operatorname{sign}(\rho)\max(|\rho|-\lambda,0)$. The residual removes the current feature before asking whether that feature still explains enough leftover signal to pay its L1 cost.

In [ ]:
X_cd_w = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.]])  # small correlated design.
y_cd_w = np.array([1.0, 1.1, 1.9, 2.8, 1.2])                                             # target.
X_cd_w = X_cd_w - X_cd_w.mean(axis=0)                                                      # center columns.
y_cd_w = y_cd_w - y_cd_w.mean()                                                           # center target.
beta_cd_w = np.zeros(3)                                                                    # start sparse.
print("column squared norms:", np.round(np.sum(X_cd_w ** 2, axis=0), 3))                  # denominators.

▶ What you'll see: each feature has its own squared length, so the thresholded score is scaled by that length.

In [ ]:
def soft_threshold_w(rho, lam):                         # L1 coordinate operator.
    return np.sign(rho) * max(abs(rho) - lam, 0.0)       # shrink toward zero, then clip at zero.

lam_cd_w = 0.45                                          # coordinate-descent penalty.
for epoch_w in range(25):                                # repeated passes over coordinates.
    for j_w in range(X_cd_w.shape[1]):                    # update one coefficient at a time.
        residual_j_w = y_cd_w - X_cd_w @ beta_cd_w + X_cd_w[:, j_w] * beta_cd_w[j_w]  # add back feature j.
        rho_j_w = float(X_cd_w[:, j_w] @ residual_j_w)    # feature j's remaining correlation.
        beta_cd_w[j_w] = soft_threshold_w(rho_j_w, lam_cd_w) / float(np.sum(X_cd_w[:, j_w] ** 2))  # update.
print("coordinate-descent beta:", np.round(beta_cd_w, 3))

▶ What you'll see: some coefficients stay or become exactly zero if their residual correlation cannot clear λ.

In [ ]:
pred_cd_w = X_cd_w @ beta_cd_w                           # fitted centered predictions.
obj_cd_w = 0.5 * float(np.sum((y_cd_w - pred_cd_w) ** 2)) + lam_cd_w * float(np.sum(np.abs(beta_cd_w)))
nnz_cd_w = int(np.sum(np.abs(beta_cd_w) > 1e-8))          # number of active coefficients.
print("objective:", round(obj_cd_w, 3), "nonzeros:", nnz_cd_w)  # inspect fit + sparsity.
assert nnz_cd_w <= 3                                      # sanity check that active count is valid.

▶ What you'll see: the final model has a finite objective and a count of selected nonzero features.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["β0", "β1", "β2"], beta_cd_w, color=["seagreen" if abs(v) > 1e-8 else "lightgray" for v in beta_cd_w])
plt.axhline(0, color="black", linewidth=0.8); plt.ylabel("coefficient"); plt.title("5: coordinate descent can select features"); plt.show()

▶ What you'll see: inactive features sit exactly on the zero line, which is the sparse feature-selection effect.

*Why it's done this way: coordinate descent turns a multivariate optimization into repeated one-dimensional Lasso problems, and each one has a closed-form soft-threshold update.*

### 6. The λ path: more penalty means more sparsity

The penalty strength $\lambda$ is a knob. At $\lambda=0$, Lasso behaves like least squares on the training objective. As $\lambda$ grows, coefficients must buy their way into the model by improving fit enough to overcome a larger threshold. Therefore the number of nonzero coefficients usually falls as λ increases.

In [ ]:
def lasso_cd_w(X, y, lam, steps=80):                    # tiny coordinate-descent solver for standardized demos.
    beta = np.zeros(X.shape[1])                         # start with all features inactive.
    norms = np.sum(X ** 2, axis=0)                      # coordinate denominators.
    for _ in range(steps):                              # repeated coordinate passes.
        for j in range(X.shape[1]):                     # update one coefficient.
            r_j = y - X @ beta + X[:, j] * beta[j]      # residual excluding current coefficient.
            rho = float(X[:, j] @ r_j)                  # remaining correlation for feature j.
            beta[j] = np.sign(rho) * max(abs(rho) - lam, 0.0) / norms[j]  # soft-threshold update.
    return beta                                         # fitted coefficients.

lams_path_w = np.array([0.0, 0.2, 0.5, 1.0, 1.8])        # penalty grid.
print("lambda grid:", lams_path_w)

▶ What you'll see: the sweep will compare no penalty through a strong penalty.

In [ ]:
betas_path_w = np.array([lasso_cd_w(X_cd_w, y_cd_w, lam) for lam in lams_path_w])  # one beta vector per lambda.
nonzeros_path_w = np.sum(np.abs(betas_path_w) > 1e-8, axis=1)                     # active feature counts.
print("betas by lambda:\n", np.round(betas_path_w, 3))
print("nonzeros:", nonzeros_path_w)
assert nonzeros_path_w[-1] <= nonzeros_path_w[0]                                  # stronger penalty no denser here.

▶ What you'll see: coefficients shrink as λ grows, and the active-feature count does not increase in this toy path.

In [ ]:
plt.figure(figsize=(5, 3.2))
for j_w in range(betas_path_w.shape[1]):
    plt.plot(lams_path_w, betas_path_w[:, j_w], marker="o", label=f"β{j_w}")
plt.axhline(0, color="black", linewidth=0.8); plt.xlabel("λ"); plt.ylabel("coefficient")
plt.title("6: Lasso coefficient path"); plt.legend(); plt.show()

▶ What you'll see: coefficient curves move toward zero as λ rises; some flatten exactly at zero.

*Why it's done this way: λ defines the price of coefficient size, so sweeping λ exposes the bias-sparsity tradeoff rather than hiding it inside one arbitrary setting.*

### 7. Validation and stability: compare full decision scores

The lesson's model-selection arithmetic compares a baseline score 0.359, a more flexible alternative score 0.411, and a stabilized score that is 20% lower than 0.359:

$$stable=0.80\cdot0.359=0.287.$$

The best score is the minimum of the full scores, not the minimum raw training fragment.

In [ ]:
alternative_w = 0.411                                      # tempting more-flexible alternative score.
stable_w = 0.80 * score_w                                  # 20% reduction from the baseline full score.
gap_w = alternative_w - score_w                            # absolute evidence gap.
relative_gap_w = gap_w / alternative_w                      # scale-aware gap.
print("baseline:", round(score_w, 3), "alternative:", alternative_w, "stable:", round(stable_w, 3))

▶ What you'll see: all three candidates are placed on the same score scale before choosing.

In [ ]:
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))  # 0.052 and 0.127.
assert round(gap_w, 3) == 0.052
assert round(relative_gap_w, 3) == 0.127
assert round(stable_w, 3) == 0.287

▶ What you'll see: the flexible alternative is worse by 0.052, about 12.7% of its own score.

In [ ]:
scores_w = np.array([score_w, alternative_w, stable_w])                 # full comparable decision scores.
labels_w = np.array(["baseline", "flexible", "stable"])
best_w = labels_w[int(np.argmin(scores_w))]                             # lower score wins.
print("winner:", best_w, "score:", round(float(scores_w.min()), 3))
assert best_w == "stable"                                                # lesson conclusion.
plt.figure(figsize=(4.6, 3))
plt.bar(labels_w, scores_w, color=["gray", "orange", "seagreen"])
plt.ylabel("full decision score"); plt.title("7: choose by full score"); plt.show()

▶ What you'll see: the stabilized candidate has the lowest full score, so it is the one to carry forward in this toy case.

*Why it's done this way: validation gaps are meaningful only when scores live on the same scale; comparing raw loss to penalized loss silently changes the rules of the contest.*


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each Lasso mechanic by hand.** These new tiny NumPy-only toys isolate
> each computational step from the walkthrough. Run them top to bottom: every block prints the
> intermediate values, draws one picture, and includes an `assert` that pins the result.

### ✍️ Toy 1 · Empirical risk averages the observed losses

Start with six per-example losses. The empirical risk is just their mean, so the largest bar pulls the dashed average upward.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_losses = np.array([0.04, 0.09, 0.16, 0.25, 0.01, 0.36])  # -> [0.04 0.09 0.16 0.25 0.01 0.36]
print("losses:", t1_losses.tolist())  # -> [0.04, 0.09, 0.16, 0.25, 0.01, 0.36]
t1_total = float(t1_losses.sum())  # -> 0.91
print("total loss:", round(t1_total, 4))  # -> 0.91
t1_risk = float(t1_losses.mean())  # -> 0.15166666666666667
print("empirical risk:", round(t1_risk, 4))  # -> 0.1517
assert abs(t1_risk - 0.91 / 6) < 1e-12

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t1_losses.size), t1_losses, color="steelblue")
plt.axhline(t1_risk, color="crimson", linestyle="--", label=f"mean={t1_risk:.4f}")
plt.xlabel("example")
plt.ylabel("loss")
plt.title("Toy 1 · empirical risk is an average")
plt.legend()
plt.show()

▶ What you'll see: six loss bars and a dashed mean line at `0.1517`.

### ✍️ Toy 2 · Regularization cost changes the score winner

A model with the smallest raw loss is not automatically best. Add the cost first, then compare full scores.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_risk = np.array([0.22, 0.18, 0.16])  # -> [0.22 0.18 0.16]
print("raw risks:", np.round(t2_risk, 3).tolist())  # -> [0.22, 0.18, 0.16]
t2_cost = np.array([0.02, 0.08, 0.14])  # -> [0.02 0.08 0.14]
print("costs:", np.round(t2_cost, 3).tolist())  # -> [0.02, 0.08, 0.14]
t2_scores = t2_risk + t2_cost  # -> [0.24 0.26 0.30]
print("full scores:", np.round(t2_scores, 3).tolist())  # -> [0.24, 0.26, 0.3]
t2_winner = int(np.argmin(t2_scores))  # -> 0
print("winner index:", t2_winner)  # -> 0
assert t2_winner == 0

plt.figure(figsize=(4.8, 2.8))
plt.bar(["A", "B", "C"], t2_scores, color=["seagreen", "gray", "gray"])
plt.ylabel("risk + cost")
plt.title("Toy 2 · compare full scores")
plt.show()

▶ What you'll see: model A wins after cost even though model C had the lowest raw risk.

### ✍️ Toy 3 · The Lasso objective adds fit and L1 size

For one candidate coefficient vector, compute predictions, residuals, half squared error, L1 norm, penalty, and total objective.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [1.0, 2.0], [2.0, 2.0]])
print("design shape:", t3_X.shape)  # -> (6, 2)
t3_y = np.array([1.0, 1.0, 2.0, 3.0, 3.0, 4.0])
print("targets:", t3_y.tolist())  # -> [1.0, 1.0, 2.0, 3.0, 3.0, 4.0]
t3_beta = np.array([0.9, 0.8])
print("beta:", t3_beta.tolist())  # -> [0.9, 0.8]
t3_pred = t3_X @ t3_beta  # -> [0.9 0.8 1.7 2.6 2.5 3.4]
print("predictions:", np.round(t3_pred, 3).tolist())  # -> [0.9, 0.8, 1.7, 2.6, 2.5, 3.4]
t3_resid = t3_y - t3_pred  # -> [0.1 0.2 0.3 0.4 0.5 0.6]
print("residuals:", np.round(t3_resid, 3).tolist())  # -> [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
t3_half_sse = 0.5 * float(np.sum(t3_resid ** 2))  # -> 0.455
print("half SSE:", round(t3_half_sse, 3))  # -> 0.455
t3_l1 = float(np.sum(np.abs(t3_beta)))  # -> 1.7
print("L1 norm:", round(t3_l1, 3))  # -> 1.7
t3_lam = 0.3
print("lambda:", t3_lam)  # -> 0.3
t3_penalty = t3_lam * t3_l1  # -> 0.51
print("lambda times L1:", round(t3_penalty, 3))  # -> 0.51
t3_objective = t3_half_sse + t3_penalty  # -> 0.965
print("Lasso objective:", round(t3_objective, 3))  # -> 0.965
assert round(t3_objective, 3) == 0.965

plt.figure(figsize=(5.0, 2.8))
plt.bar(["0.5 SSE", "λ||β||₁", "total"], [t3_half_sse, t3_penalty, t3_objective], color=["steelblue", "orange", "purple"])
plt.ylabel("objective value")
plt.title("Toy 3 · fit term plus L1 cost")
plt.show()

▶ What you'll see: the total objective bar is the sum of the fit bar and the L1 penalty bar.

### ✍️ Toy 4 · Soft-thresholding creates exact zeros

The L1 corner turns weak coordinate signals into exact zeros while shrinking stronger signals toward zero.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_z = np.array([-1.2, -0.5, -0.2, 0.0, 0.3, 0.8, 1.5])
print("unpenalized z:", t4_z.tolist())  # -> [-1.2, -0.5, -0.2, 0.0, 0.3, 0.8, 1.5]
t4_lam = 0.4
print("lambda:", t4_lam)  # -> 0.4
t4_soft_raw = np.sign(t4_z) * np.maximum(np.abs(t4_z) - t4_lam, 0.0)
t4_soft = np.where(np.isclose(t4_soft_raw, 0.0), 0.0, t4_soft_raw)  # -> [-0.8 -0.1  0.   0.   0.   0.4  1.1]
print("soft-thresholded:", np.round(t4_soft, 3).tolist())  # -> [-0.8, -0.1, 0.0, 0.0, 0.0, 0.4, 1.1]
t4_zero_mask = np.isclose(t4_soft, 0.0)  # -> [False False  True  True  True False False]
print("zero mask:", t4_zero_mask.astype(int).tolist())  # -> [0, 0, 1, 1, 1, 0, 0]
t4_nonzeros = int(np.sum(~t4_zero_mask))  # -> 4
print("nonzero count:", t4_nonzeros)  # -> 4
assert np.allclose(t4_soft, [-0.8, -0.1, 0.0, 0.0, 0.0, 0.4, 1.1])

plt.figure(figsize=(5.0, 2.8))
plt.bar(np.arange(t4_z.size), t4_soft, color=["lightgray" if z else "seagreen" for z in t4_zero_mask])
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("coordinate")
plt.ylabel("new coefficient")
plt.title("Toy 4 · weak signals become zero")
plt.show()

▶ What you'll see: the middle three coordinates sit exactly on the zero line.

### ✍️ Toy 5 · One coordinate update uses a partial residual

Coordinate descent updates one coefficient, recomputes the residual for the next coefficient, and soft-thresholds that coordinate's remaining correlation.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_x1 = np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
print("feature 1:", t5_x1.tolist())  # -> [-3.0, -2.0, -1.0, 1.0, 2.0, 3.0]
t5_x2 = np.array([-1.0, 1.0, -1.0, 1.0, -1.0, 1.0])
print("feature 2:", t5_x2.tolist())  # -> [-1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
t5_X = np.column_stack([t5_x1, t5_x2])
print("design shape:", t5_X.shape)  # -> (6, 2)
t5_y = np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
print("target:", t5_y.tolist())  # -> [-3.0, -2.0, -1.0, 1.0, 2.0, 3.0]
t5_beta = np.zeros(2)
print("start beta:", t5_beta.tolist())  # -> [0.0, 0.0]
t5_lam = 1.0
print("lambda:", t5_lam)  # -> 1.0
t5_norms = np.sum(t5_X ** 2, axis=0)  # -> [28.  6.]
print("column norms:", np.round(t5_norms, 3).tolist())  # -> [28.0, 6.0]
t5_resid0 = t5_y - t5_X @ t5_beta + t5_X[:, 0] * t5_beta[0]
t5_rho0 = float(t5_X[:, 0] @ t5_resid0)  # -> 28.0
print("rho for feature 1:", round(t5_rho0, 3))  # -> 28.0
t5_update0 = np.sign(t5_rho0) * max(abs(t5_rho0) - t5_lam, 0.0) / float(t5_norms[0])  # -> 0.9642857142857143
print("feature 1 update:", round(t5_update0, 3))  # -> 0.964
t5_beta[0] = t5_update0
print("beta after feature 1:", np.round(t5_beta, 3).tolist())  # -> [0.964, 0.0]
t5_resid1 = t5_y - t5_X @ t5_beta + t5_X[:, 1] * t5_beta[1]
t5_rho1 = float(t5_X[:, 1] @ t5_resid1)  # -> 0.1428571428571428
print("rho for feature 2:", round(t5_rho1, 3))  # -> 0.143
t5_update1 = np.sign(t5_rho1) * max(abs(t5_rho1) - t5_lam, 0.0) / float(t5_norms[1])  # -> 0.0
print("feature 2 update:", round(t5_update1, 3))  # -> 0.0
t5_beta[1] = t5_update1
print("beta after feature 2:", np.round(t5_beta, 3).tolist())  # -> [0.964, 0.0]
t5_pred = t5_X @ t5_beta  # -> [-2.89285714 -1.92857143 -0.96428571  0.96428571  1.92857143  2.89285714]
print("predictions:", np.round(t5_pred, 3).tolist())  # -> [-2.893, -1.929, -0.964, 0.964, 1.929, 2.893]
t5_objective = 0.5 * float(np.sum((t5_y - t5_pred) ** 2)) + t5_lam * float(np.sum(np.abs(t5_beta)))  # -> 0.9821428571428572
print("objective:", round(t5_objective, 3))  # -> 0.982
assert abs(t5_beta[1]) < 1e-12

plt.figure(figsize=(4.8, 2.8))
plt.bar(["β1", "β2"], t5_beta, color=["seagreen", "lightgray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("coefficient")
plt.title("Toy 5 · coordinate 2 cannot clear λ")
plt.show()

▶ What you'll see: the first feature enters, while the second feature is thresholded to exactly zero.

### ✍️ Toy 6 · A lambda path shrinks active features

Fit the same two-feature toy across a few `lambda` values. As the threshold grows, coefficients shrink and the active count falls.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_x1 = np.array([-2.0, -1.0, 0.0, 0.0, 1.0, 2.0])
print("feature 1:", t6_x1.tolist())  # -> [-2.0, -1.0, 0.0, 0.0, 1.0, 2.0]
t6_x2 = np.array([-1.0, 1.0, -1.0, 1.0, -1.0, 1.0])
print("feature 2:", t6_x2.tolist())  # -> [-1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
t6_X = np.column_stack([t6_x1, t6_x2])
print("design shape:", t6_X.shape)  # -> (6, 2)
t6_y = t6_x1 + 0.4 * t6_x2  # -> [-2.4 -0.6 -0.4  0.4  0.6  2.4]
print("target:", np.round(t6_y, 3).tolist())  # -> [-2.4, -0.6, -0.4, 0.4, 0.6, 2.4]
t6_lams = np.array([0.0, 0.5, 1.5, 3.0])
print("lambda grid:", t6_lams.tolist())  # -> [0.0, 0.5, 1.5, 3.0]

def t6_fit_lasso(X, y, lam, steps=60):
    beta = np.zeros(X.shape[1])
    norms = np.sum(X ** 2, axis=0)
    for _ in range(steps):
        for j in range(X.shape[1]):
            residual = y - X @ beta + X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            beta[j] = np.sign(rho) * max(abs(rho) - lam, 0.0) / float(norms[j])
    return beta

t6_betas = np.array([t6_fit_lasso(t6_X, t6_y, lam) for lam in t6_lams])  # -> [[1.0, 0.4], [0.964, 0.329], [0.893, 0.186], [0.78, 0.0]]
print("betas by lambda:", np.round(t6_betas, 3).tolist())  # -> [[1.0, 0.4], [0.964, 0.329], [0.893, 0.186], [0.78, 0.0]]
t6_nonzeros = np.sum(np.abs(t6_betas) > 1e-8, axis=1)  # -> [2 2 2 1]
print("nonzero counts:", t6_nonzeros.tolist())  # -> [2, 2, 2, 1]
assert np.all(np.diff(t6_nonzeros) <= 0)

plt.figure(figsize=(5.0, 2.8))
plt.plot(t6_lams, t6_betas[:, 0], marker="o", label="β1")
plt.plot(t6_lams, t6_betas[:, 1], marker="o", label="β2")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("λ")
plt.ylabel("coefficient")
plt.title("Toy 6 · larger λ gives a sparser path")
plt.legend()
plt.show()

▶ What you'll see: both coefficient curves shrink, and `β2` lands exactly at zero for the largest `λ`.

### ✍️ Toy 7 · Full decision scores beat raw-fit temptation

The flexible option has the prettiest raw fit, but the stabilized option wins after costs and a stability discount are put on the same score scale.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_labels = np.array(["baseline", "flexible", "stable"])
print("labels:", t7_labels.tolist())  # -> ['baseline', 'flexible', 'stable']
t7_raw = np.array([0.21, 0.15, 0.20])
print("raw risks:", np.round(t7_raw, 3).tolist())  # -> [0.21, 0.15, 0.2]
t7_cost = np.array([0.04, 0.11, 0.02])
print("costs:", np.round(t7_cost, 3).tolist())  # -> [0.04, 0.11, 0.02]
t7_discount = np.array([0.00, 0.00, 0.06])
print("stability discounts:", np.round(t7_discount, 3).tolist())  # -> [0.0, 0.0, 0.06]
t7_scores = t7_raw + t7_cost - t7_discount  # -> [0.25 0.26 0.16]
print("full scores:", np.round(t7_scores, 3).tolist())  # -> [0.25, 0.26, 0.16]
t7_best = int(np.argmin(t7_scores))  # -> 2
print("full-score winner:", t7_labels[t7_best])  # -> stable
t7_raw_best = int(np.argmin(t7_raw))  # -> 1
print("raw-risk winner:", t7_labels[t7_raw_best])  # -> flexible
t7_gap = float(t7_scores[1] - t7_scores[t7_best])  # -> 0.1
print("flexible minus stable gap:", round(t7_gap, 3))  # -> 0.1
assert t7_labels[t7_best] == "stable" and t7_labels[t7_raw_best] == "flexible"

plt.figure(figsize=(5.0, 2.8))
plt.bar(t7_labels, t7_scores, color=["gray", "orange", "seagreen"])
plt.ylabel("full decision score")
plt.title("Toy 7 · choose after cost and stability")
plt.show()

▶ What you'll see: the green stabilized bar is lowest even though the orange flexible model had the lowest raw risk.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, linear algebra, random numbers, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib so each Lasso idea can be inspected visually.
np.random.seed(0)  # make all examples reproducible across notebook runs.

def l1_norm(beta):  # compute the L1 size of a coefficient vector.
    beta = np.asarray(beta, dtype=float)  # convert input to predictable floating-point values.
    return float(np.sum(np.abs(beta)))  # sum absolute values, the sparsity-producing penalty.

def lasso_objective(X, y, beta, lam):  # compute 0.5||y-Xb||^2 + lambda||b||_1.
    X = np.asarray(X, dtype=float)  # convert features to a numeric array.
    y = np.asarray(y, dtype=float)  # convert targets to a numeric array.
    beta = np.asarray(beta, dtype=float)  # convert coefficients to a numeric array.
    resid = y - X @ beta  # compute residuals from the linear predictions.
    return 0.5 * float(np.sum(resid ** 2)) + lam * l1_norm(beta)  # combine fit cost and L1 cost.

def soft_threshold(z, lam):  # define the Lasso one-coordinate solution S(z, lambda).
    z = np.asarray(z, dtype=float)  # allow scalar or vector input.
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)  # shrink toward zero and clip small values to exact zero.

def lasso_cd(X, y, lam, steps=100):  # small coordinate-descent Lasso solver for centered toy data.
    X = np.asarray(X, dtype=float)  # convert features to floats.
    y = np.asarray(y, dtype=float)  # convert targets to floats.
    beta = np.zeros(X.shape[1])  # start from the sparsest coefficient vector.
    norms = np.sum(X ** 2, axis=0)  # precompute feature squared norms for coordinate denominators.
    for _ in range(steps):  # make repeated passes over all coordinates.
        for j in range(X.shape[1]):  # update one coefficient at a time.
            r_j = y - X @ beta + X[:, j] * beta[j]  # partial residual with feature j removed.
            rho = float(X[:, j] @ r_j)  # correlation between feature j and leftover signal.
            beta[j] = soft_threshold(rho, lam) / norms[j] if norms[j] > 0 else 0.0  # closed-form L1 update.
    return beta  # return the fitted sparse coefficient vector.

def standardize(X):  # center and scale features for stable coordinate descent.
    X = np.asarray(X, dtype=float)  # convert to float array.
    mu = X.mean(axis=0)  # feature means.
    sigma = X.std(axis=0)  # feature standard deviations.
    sigma = np.where(sigma == 0, 1.0, sigma)  # avoid division by zero for constant columns.
    return (X - mu) / sigma  # standardized design matrix.

## 🟢 Basics (warm-up)

### Basic 1 — Average three training losses

**Goal.** Compute the empirical risk from the lesson's three verified losses, because Lasso still begins with an average training fit term before any penalty is added. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.180, 0.135, 0.522])  # store the three per-example losses from the lesson prose.
print("losses_b1:", losses_b1)  # inspect the pieces that will be averaged.
print("count_b1:", losses_b1.size)  # inspect the denominator of the empirical risk.

▶ What you'll see: three individual losses and a count of 3.

In [ ]:
risk_b1 = float(losses_b1.mean())  # average the losses to get empirical risk R_S.
print("R_S_b1:", round(risk_b1, 3))  # inspect the verified risk.
assert round(risk_b1, 3) == 0.279  # verify the lesson arithmetic.
plt.figure(figsize=(4, 3))  # create a compact loss bar chart.
plt.bar(["1", "2", "3"], losses_b1, color="steelblue")  # show each example loss.
plt.axhline(risk_b1, color="red", linestyle="--", label="mean")  # show the empirical risk.
plt.title("Basic 1: empirical risk")  # title the diagnostic plot.
plt.ylabel("loss")  # label the loss scale.
plt.legend()  # show the mean label.
plt.show()  # display the figure.

▶ What you'll see: the mean line sits at 0.279 across the three training losses.

👀 Takeaway: empirical risk is an average over examples, not the smallest or largest single loss.

### Basic 2 — Add the regularization cost

**Goal.** Add the lesson's 0.080 cost to the empirical risk, because model selection should compare the full score rather than the raw fit alone. We build it in 2 steps.

In [ ]:
risk_b2 = 0.279  # reuse the verified empirical risk from the lesson.
cost_b2 = 0.080  # store the regularization or complexity cost.
print("risk_b2:", risk_b2, "cost_b2:", cost_b2)  # inspect the two pieces before adding.

▶ What you'll see: the score has a fit term and a separate cost term.

In [ ]:
score_b2 = risk_b2 + cost_b2  # combine fit and cost into the decision score.
print("score_b2:", round(score_b2, 3))  # inspect the full score.
assert round(score_b2, 3) == 0.359  # verify the lesson number.
plt.figure(figsize=(4, 3))  # create a compact decomposition plot.
plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["green", "orange", "purple"])  # visualize the pieces.
plt.title("Basic 2: risk + cost")  # title the plot.
plt.ylabel("value")  # label the value scale.
plt.show()  # display the chart.

▶ What you'll see: the full score is higher than the raw risk because the model pays for complexity.

👀 Takeaway: regularization changes which number is eligible for model comparison.

### Basic 3 — Compute an L1 norm

**Goal.** Sum absolute coefficient values, because Lasso's penalty is λ times the L1 norm. We build it in 2 steps.

In [ ]:
beta_b3 = np.array([1.5, -0.4, 0.0, 2.1])  # define a coefficient vector with positive, negative, and zero entries.
abs_b3 = np.abs(beta_b3)  # take absolute values because L1 charges magnitude, not sign.
print("beta_b3:", beta_b3)  # inspect the original coefficients.
print("abs_b3:", abs_b3)  # inspect the values that enter the penalty.

▶ What you'll see: negative coefficients become positive costs while exact zeros add nothing.

In [ ]:
l1_b3 = l1_norm(beta_b3)  # sum absolute coefficients.
print("L1 norm_b3:", round(l1_b3, 3))  # inspect the L1 size.
assert round(l1_b3, 3) == 4.0  # 1.5 + 0.4 + 0 + 2.1.
plt.figure(figsize=(4, 3))  # create a magnitude plot.
plt.bar(["β0", "β1", "β2", "β3"], abs_b3, color="teal")  # visualize penalty contributions.
plt.title("Basic 3: absolute coefficient costs")  # title the plot.
plt.ylabel("|β_j|")  # label the L1 contribution scale.
plt.show()  # display the chart.

▶ What you'll see: each bar is a nonnegative amount paid into the L1 penalty.

👀 Takeaway: L1 penalizes coefficient magnitude and is indifferent to coefficient sign.

### Basic 4 — Compute squared-error fit

**Goal.** Measure how well one linear coefficient vector predicts y, because the Lasso objective combines this fit term with L1 size. We build it in 2 steps.

In [ ]:
X_b4 = np.array([[1., 0.], [0., 1.], [1., 1.], [2., 1.]])  # define four examples and two features.
y_b4 = np.array([1.0, 1.0, 2.0, 2.5])  # define target values.
beta_b4 = np.array([0.8, 0.6])  # choose one candidate coefficient vector.
pred_b4 = X_b4 @ beta_b4  # compute linear predictions.
print("pred_b4:", np.round(pred_b4, 3))  # inspect predictions before scoring.

▶ What you'll see: each prediction is a row of X multiplied by β.

In [ ]:
resid_b4 = y_b4 - pred_b4  # compute residuals.
half_sse_b4 = 0.5 * float(np.sum(resid_b4 ** 2))  # compute half squared error.
print("residuals_b4:", np.round(resid_b4, 3))  # inspect misses.
print("half SSE_b4:", round(half_sse_b4, 3))  # inspect the fit term.
assert round(half_sse_b4, 3) == 0.325  # verify this toy calculation.
plt.figure(figsize=(4, 3))  # create a residual plot.
plt.bar(range(len(resid_b4)), resid_b4, color="slateblue")  # show signed residuals.
plt.axhline(0, color="black", linewidth=0.8)  # mark perfect prediction.
plt.title("Basic 4: residuals behind squared error")  # title the plot.
plt.ylabel("y - Xβ")  # label residual scale.
plt.show()  # display the figure.

▶ What you'll see: positive and negative residuals both become positive after squaring.

👀 Takeaway: squared error rewards predictions close to y regardless of residual sign.

### Basic 5 — Combine fit and L1 into the Lasso objective

**Goal.** Compute the full Lasso objective for one β, because optimization compares fit plus λ times coefficient size. We build it in 3 steps.

In [ ]:
X_b5 = np.array([[1., 0.], [0., 1.], [1., 1.], [2., 1.]])  # recreate the tiny design matrix locally.
y_b5 = np.array([1.0, 1.0, 2.0, 2.5])  # recreate targets locally.
beta_b5 = np.array([0.8, 0.6])  # choose the same candidate vector.
lam_b5 = 0.4  # set the L1 penalty strength.
print("lambda_b5:", lam_b5, "beta_b5:", beta_b5)  # inspect objective inputs.

▶ What you'll see: the penalty strength and candidate coefficients are explicit.

In [ ]:
fit_b5 = 0.5 * float(np.sum((y_b5 - X_b5 @ beta_b5) ** 2))  # compute the fit term.
penalty_b5 = lam_b5 * l1_norm(beta_b5)  # compute λ||β||_1.
print("fit_b5:", round(fit_b5, 3), "penalty_b5:", round(penalty_b5, 3))  # inspect objective pieces.

▶ What you'll see: the penalty can be as large as, or larger than, the residual fit term.

In [ ]:
obj_b5 = lasso_objective(X_b5, y_b5, beta_b5, lam_b5)  # compute the full objective with the helper.
print("objective_b5:", round(obj_b5, 3))  # inspect the total score for this β.
assert round(obj_b5, 3) == 0.885  # verify 0.325 + 0.560.
plt.figure(figsize=(4, 3))  # create a component plot.
plt.bar(["fit", "λL1", "total"], [fit_b5, penalty_b5, obj_b5], color=["steelblue", "orange", "purple"])  # compare pieces.
plt.title("Basic 5: Lasso objective")  # title the plot.
plt.ylabel("value")  # label objective scale.
plt.show()  # display the chart.

▶ What you'll see: the objective is the total bar, not either component in isolation.

👀 Takeaway: a coefficient vector is good only if its fit improvement justifies its L1 cost.

### Basic 6 — Apply soft-thresholding by hand

**Goal.** Map unpenalized coordinate scores to Lasso coordinate values, because soft-thresholding is the mechanism that creates exact zeros. We build it in 2 steps.

In [ ]:
z_b6 = np.array([-1.4, -0.4, 0.0, 0.4, 1.4])  # define possible unpenalized coordinate scores.
lam_b6 = 0.7  # define the threshold width.
print("z_b6:", z_b6)  # inspect coordinate scores before thresholding.
print("lambda_b6:", lam_b6)  # inspect the cutoff.

▶ What you'll see: scores with magnitude below 0.7 should be killed by the L1 penalty.

In [ ]:
soft_b6 = soft_threshold(z_b6, lam_b6)  # apply the Lasso coordinate operator.
print("soft_b6:", soft_b6)  # inspect exact zeros and shrunken survivors.
assert np.allclose(soft_b6, [-0.7, 0.0, 0.0, 0.0, 0.7])  # verify the thresholding result.
plt.figure(figsize=(4, 3))  # create a before-after plot.
plt.plot(z_b6, z_b6, "o--", label="before z", color="gray")  # show unpenalized coordinates.
plt.plot(z_b6, soft_b6, "o-", label="after S(z,λ)", color="purple")  # show thresholded coordinates.
plt.axhline(0, color="black", linewidth=0.8)  # mark exact zero.
plt.title("Basic 6: soft-thresholding")  # title the plot.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: the middle three values collapse onto the zero line.

👀 Takeaway: Lasso can set coefficients exactly to zero because the threshold has a flat zero region.

### Basic 7 — Count nonzero coefficients

**Goal.** Measure sparsity by counting active coefficients, because Lasso's interpretability often comes from selecting only a few features. We build it in 2 steps.

In [ ]:
beta_b7 = np.array([0.8, 0.0, -0.2, 0.0, 1.1])  # define a sparse coefficient vector.
active_b7 = np.abs(beta_b7) > 1e-12  # mark coefficients that are numerically nonzero.
print("active mask_b7:", active_b7.astype(int))  # inspect selected features.

▶ What you'll see: ones mark selected features and zeros mark dropped features.

In [ ]:
nnz_b7 = int(np.sum(active_b7))  # count active coefficients.
sparsity_b7 = 1.0 - nnz_b7 / beta_b7.size  # compute fraction of coefficients equal to zero.
print("nonzeros_b7:", nnz_b7, "sparsity_b7:", round(sparsity_b7, 3))  # inspect model sparsity.
assert nnz_b7 == 3  # verify active count.
plt.figure(figsize=(4, 3))  # create a coefficient plot.
plt.bar([f"β{i}" for i in range(beta_b7.size)], beta_b7, color=["seagreen" if a else "lightgray" for a in active_b7])  # color selected features.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.title("Basic 7: active coefficients")  # title the plot.
plt.ylabel("coefficient")  # label coefficient scale.
plt.show()  # display the chart.

▶ What you'll see: inactive coefficients sit exactly at zero and do not participate in prediction.

👀 Takeaway: sparsity is the fraction or count of coefficients that Lasso drives to zero.

### Basic 8 — Compare L1 and L2 shrinkage on one coordinate

**Goal.** Contrast soft-thresholding with ridge-style shrinkage, because L1 can produce exact zeros while L2 usually only makes values smaller. We build it in 3 steps.

In [ ]:
z_b8 = np.linspace(-2.0, 2.0, 9)  # define coordinate scores across negative and positive values.
lam_b8 = 0.8  # use the same strength for a visual comparison.
print("z grid_b8:", np.round(z_b8, 2))  # inspect the coordinate scores.

▶ What you'll see: the grid includes values inside and outside the L1 zero region.

In [ ]:
l1_sol_b8 = soft_threshold(z_b8, lam_b8)  # Lasso coordinate solution.
l2_sol_b8 = z_b8 / (1.0 + lam_b8)  # ridge-style coordinate shrinkage for a comparable quadratic penalty.
print("L1_b8:", np.round(l1_sol_b8, 3))  # inspect exact-zero behavior.
print("L2_b8:", np.round(l2_sol_b8, 3))  # inspect smooth shrinkage.

▶ What you'll see: L1 has several exact zeros, while L2 keeps nonzero values except when z itself is zero.

In [ ]:
plt.figure(figsize=(4.8, 3))  # create a shrinkage comparison plot.
plt.plot(z_b8, l1_sol_b8, marker="o", label="L1 soft", color="purple")  # plot L1 solution.
plt.plot(z_b8, l2_sol_b8, marker="s", label="L2 shrink", color="teal")  # plot L2-style solution.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.title("Basic 8: L1 vs L2 shrinkage")  # title the plot.
plt.xlabel("unpenalized z")  # label input coordinate.
plt.ylabel("penalized coefficient")  # label output coordinate.
plt.legend()  # show curve labels.
plt.show()  # display the comparison.

▶ What you'll see: L1 has a flat zero plateau that L2 does not have.

👀 Takeaway: both penalties shrink, but only L1 naturally performs exact feature selection.

### Basic 9 — Predict with a sparse linear model

**Goal.** Use only nonzero coefficients to make predictions, because dropped Lasso features contribute exactly nothing to Xβ. We build it in 2 steps.

In [ ]:
X_b9 = np.array([[1., 2., 0.], [0., 1., 3.], [2., 0., 1.]])  # define three examples with three features.
beta_b9 = np.array([1.5, 0.0, -0.5])  # define a sparse coefficient vector.
active_b9 = np.where(np.abs(beta_b9) > 1e-12)[0]  # find selected feature indices.
print("active feature indices_b9:", active_b9)  # inspect which columns matter.

▶ What you'll see: feature 1 is absent because its coefficient is exactly zero.

In [ ]:
pred_b9 = X_b9 @ beta_b9  # compute predictions using all columns; zero coefficients contribute nothing.
pred_active_b9 = X_b9[:, active_b9] @ beta_b9[active_b9]  # compute the same predictions from active columns only.
print("pred_b9:", np.round(pred_b9, 3))  # inspect model predictions.
assert np.allclose(pred_b9, pred_active_b9)  # verify dropped columns can be removed.
plt.figure(figsize=(4, 3))  # create a prediction plot.
plt.bar(["ex0", "ex1", "ex2"], pred_b9, color="darkcyan")  # show predicted values.
plt.title("Basic 9: sparse model predictions")  # title the plot.
plt.ylabel("Xβ")  # label prediction scale.
plt.show()  # display the chart.

▶ What you'll see: predictions are identical whether the zero-coefficient feature is present or removed.

👀 Takeaway: exact zeros make Lasso models cheaper and easier to interpret because unused features vanish from prediction.

### Basic 10 — Choose the lowest full score

**Goal.** Reproduce the lesson's final score comparison, because the correct winner is the lowest full decision score on a common scale. We build it in 3 steps.

In [ ]:
baseline_b10 = 0.359  # store the baseline full score.
flexible_b10 = 0.411  # store the tempting alternative's full score.
stable_b10 = 0.80 * baseline_b10  # apply the lesson's 20% stability improvement.
print("scores before rounding_b10:", baseline_b10, flexible_b10, stable_b10)  # inspect candidates.

▶ What you'll see: three comparable full scores, not a mix of raw and penalized quantities.

In [ ]:
gap_b10 = flexible_b10 - baseline_b10  # absolute gap between flexible and baseline.
rel_gap_b10 = gap_b10 / flexible_b10  # relative gap on the flexible model's scale.
print("gap_b10:", round(gap_b10, 3), "relative_b10:", round(rel_gap_b10, 3))  # inspect evidence size.
assert round(gap_b10, 3) == 0.052  # verify lesson gap.
assert round(rel_gap_b10, 3) == 0.127  # verify lesson relative gap.

▶ What you'll see: the flexible alternative is worse by 0.052, or about 12.7% of its score.

In [ ]:
scores_b10 = np.array([baseline_b10, flexible_b10, stable_b10])  # collect candidate scores.
labels_b10 = np.array(["baseline", "flexible", "stable"])  # label candidates.
winner_b10 = labels_b10[int(np.argmin(scores_b10))]  # choose the minimum score.
print("winner_b10:", winner_b10, "score:", round(float(scores_b10.min()), 3))  # inspect the selected model.
assert winner_b10 == "stable"  # verify the lesson decision.
plt.figure(figsize=(4, 3))  # create a selection plot.
plt.bar(labels_b10, scores_b10, color=["gray", "orange", "green"])  # visualize all candidates.
plt.title("Basic 10: lower full score wins")  # title the plot.
plt.ylabel("decision score")  # label score scale.
plt.show()  # display the chart.

▶ What you'll see: the stabilized candidate has the lowest score and therefore wins this toy comparison.

👀 Takeaway: model selection must compare complete scores on the same scale.

## 🟡 Easy

### Easy 1 — Fit Lasso by coordinate descent

**Goal.** Train a tiny sparse linear model with coordinate descent, because Lasso is easiest to understand as repeated soft-threshold updates. We build it in 4 steps.

In [ ]:
X_e1_raw = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.]])  # define a small design matrix.
y_e1_raw = np.array([1.0, 1.1, 1.9, 2.8, 1.2])  # define target values.
X_e1 = standardize(X_e1_raw)  # standardize features so λ treats coordinates fairly.
y_e1 = y_e1_raw - y_e1_raw.mean()  # center the target so no intercept is needed.
print("X_e1 shape:", X_e1.shape)  # inspect data dimensions.

▶ What you'll see: a five-example, three-feature toy regression problem.

In [ ]:
lam_e1 = 0.5  # choose a moderate L1 penalty.
beta_e1 = lasso_cd(X_e1, y_e1, lam_e1, steps=120)  # fit Lasso with coordinate descent.
print("beta_e1:", np.round(beta_e1, 3))  # inspect fitted coefficients.

▶ What you'll see: a sparse or shrunken coefficient vector after repeated soft-thresholding.

In [ ]:
pred_e1 = X_e1 @ beta_e1  # compute centered predictions.
obj_e1 = lasso_objective(X_e1, y_e1, beta_e1, lam_e1)  # compute the fitted objective.
nonzero_e1 = int(np.sum(np.abs(beta_e1) > 1e-8))  # count active features.
print("objective_e1:", round(obj_e1, 3), "nonzeros_e1:", nonzero_e1)  # inspect fit and sparsity.
assert nonzero_e1 <= X_e1.shape[1]  # sanity-check active feature count.

▶ What you'll see: the final objective and the number of selected features.

In [ ]:
plt.figure(figsize=(4, 3))  # create a coefficient plot.
plt.bar(["β0", "β1", "β2"], beta_e1, color=["seagreen" if abs(v) > 1e-8 else "lightgray" for v in beta_e1])  # color active features.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.title("Easy 1: coordinate-descent Lasso")  # title the plot.
plt.ylabel("coefficient")  # label coefficient scale.
plt.show()  # display the chart.

▶ What you'll see: features that cannot pay the L1 cost are shrunk to, or held at, zero.

👀 Takeaway: coordinate descent implements Lasso by repeatedly asking each feature whether its residual correlation clears λ.

### Easy 2 — Sweep λ and count selected features

**Goal.** Fit the same data for several λ values, because λ controls the sparsity-flexibility tradeoff. We build it in 4 steps.

In [ ]:
X_e2_raw = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.]])  # recreate the design matrix.
y_e2_raw = np.array([1.0, 1.1, 1.9, 2.8, 1.2])  # recreate targets.
X_e2 = standardize(X_e2_raw)  # standardize feature scales.
y_e2 = y_e2_raw - y_e2_raw.mean()  # center target values.
print("prepared data_e2:", X_e2.shape)  # inspect dimensions.

▶ What you'll see: the λ sweep uses the same standardized data for every model.

In [ ]:
lams_e2 = np.array([0.0, 0.2, 0.5, 1.0, 1.8])  # define penalty strengths.
betas_e2 = np.array([lasso_cd(X_e2, y_e2, lam, steps=150) for lam in lams_e2])  # fit one model per λ.
print("betas_e2:\n", np.round(betas_e2, 3))  # inspect coefficient paths.

▶ What you'll see: coefficients generally shrink as λ increases.

In [ ]:
nnz_e2 = np.sum(np.abs(betas_e2) > 1e-8, axis=1)  # count selected features for each λ.
print("nonzeros_e2:", nnz_e2)  # inspect sparsity path.
assert nnz_e2[-1] <= nnz_e2[0]  # verify stronger penalty is no denser in this toy sweep.

▶ What you'll see: the active-feature count falls or stays flat as the penalty gets stronger.

In [ ]:
plt.figure(figsize=(5, 3))  # create a sparsity path plot.
plt.plot(lams_e2, nnz_e2, marker="o", color="crimson")  # plot active count by λ.
plt.title("Easy 2: selected features by λ")  # title the plot.
plt.xlabel("λ")  # label penalty axis.
plt.ylabel("nonzero coefficients")  # label active-feature count.
plt.ylim(-0.1, X_e2.shape[1] + 0.2)  # keep the count scale readable.
plt.show()  # display the line chart.

▶ What you'll see: larger λ values create sparser models.

👀 Takeaway: λ is the knob that trades training flexibility for sparsity and stability.

### Easy 3 — Compare train and validation error

**Goal.** Hold out examples and compare λ values on validation error, because the prettiest training objective may not be the best future decision. We build it in 5 steps.

In [ ]:
X_e3_raw = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.], [2., 2., 0.]])  # six examples.
y_e3_raw = np.array([1.0, 1.1, 1.9, 2.8, 1.2, 3.0])  # targets for validation demo.
train_idx_e3 = np.array([0, 1, 2, 3])  # training rows.
val_idx_e3 = np.array([4, 5])  # held-out validation rows.
print("train rows_e3:", train_idx_e3, "val rows_e3:", val_idx_e3)  # inspect split.

▶ What you'll see: the model will train on four rows and choose λ using two held-out rows.

In [ ]:
X_all_e3 = standardize(X_e3_raw)  # standardize using this toy full matrix for a compact demo.
y_all_e3 = y_e3_raw - y_e3_raw[train_idx_e3].mean()  # center around the training target mean.
X_train_e3, y_train_e3 = X_all_e3[train_idx_e3], y_all_e3[train_idx_e3]  # training data.
X_val_e3, y_val_e3 = X_all_e3[val_idx_e3], y_all_e3[val_idx_e3]  # validation data.
print("X_train_e3 shape:", X_train_e3.shape)  # inspect training shape.

▶ What you'll see: the train and validation matrices have the same feature columns.

In [ ]:
lams_e3 = np.array([0.0, 0.2, 0.6, 1.2])  # choose a small penalty grid.
train_rmse_e3 = []  # store training RMSE.
val_rmse_e3 = []  # store validation RMSE.
for lam_e3 in lams_e3:  # fit each candidate penalty.
    beta_e3 = lasso_cd(X_train_e3, y_train_e3, lam_e3, steps=180)  # train on the training rows only.
    train_rmse_e3.append(float(np.sqrt(np.mean((y_train_e3 - X_train_e3 @ beta_e3) ** 2))))  # training error.
    val_rmse_e3.append(float(np.sqrt(np.mean((y_val_e3 - X_val_e3 @ beta_e3) ** 2))))  # validation error.
print("train RMSE_e3:", np.round(train_rmse_e3, 3))  # inspect fit quality.
print("val RMSE_e3:", np.round(val_rmse_e3, 3))  # inspect future-quality proxy.

▶ What you'll see: training and validation curves need not choose the same λ.

In [ ]:
best_idx_e3 = int(np.argmin(val_rmse_e3))  # choose by validation error.
best_lam_e3 = float(lams_e3[best_idx_e3])  # read selected λ.
print("best lambda_e3:", best_lam_e3)  # inspect selected penalty.
assert best_lam_e3 in lams_e3  # verify selection comes from the grid.

▶ What you'll see: the chosen λ is the one with the lowest held-out RMSE, not necessarily the lowest training RMSE.

In [ ]:
plt.figure(figsize=(5, 3))  # create train-validation comparison plot.
plt.plot(lams_e3, train_rmse_e3, marker="o", label="train")  # plot training RMSE.
plt.plot(lams_e3, val_rmse_e3, marker="s", label="validation")  # plot validation RMSE.
plt.axvline(best_lam_e3, color="red", linestyle="--", label="best λ")  # mark selected λ.
plt.title("Easy 3: choose λ by validation")  # title the plot.
plt.xlabel("λ")  # label penalty axis.
plt.ylabel("RMSE")  # label error axis.
plt.legend()  # show curve labels.
plt.show()  # display the chart.

▶ What you'll see: the validation curve is the decision curve for model selection.

👀 Takeaway: validation keeps the Lasso penalty tied to future behavior rather than training convenience.

### Easy 4 — Show feature selection on correlated predictors

**Goal.** Fit Lasso when two features carry overlapping information, because L1 often picks one representative from a correlated group. We build it in 4 steps.

In [ ]:
x0_e4 = np.linspace(-2, 2, 20)  # create a base feature.
x1_e4 = x0_e4 + 0.15 * np.sin(np.arange(20))  # create a strongly correlated feature.
x2_e4 = np.cos(np.arange(20))  # create a less relevant feature.
X_e4 = standardize(np.column_stack([x0_e4, x1_e4, x2_e4]))  # build and standardize the design.
y_e4 = 2.0 * x0_e4 + 0.1 * x2_e4  # target mostly depends on the correlated x0/x1 signal.
y_e4 = y_e4 - y_e4.mean()  # center target.
print("corr x0,x1_e4:", round(float(np.corrcoef(X_e4[:, 0], X_e4[:, 1])[0, 1]), 3))  # inspect correlation.

▶ What you'll see: the first two predictors are highly correlated, so they compete to explain the same signal.

In [ ]:
beta_low_e4 = lasso_cd(X_e4, y_e4, lam=0.05, steps=200)  # fit with a weak penalty.
beta_high_e4 = lasso_cd(X_e4, y_e4, lam=2.0, steps=200)  # fit with a stronger penalty.
print("weak λ beta_e4:", np.round(beta_low_e4, 3))  # inspect low-penalty model.
print("strong λ beta_e4:", np.round(beta_high_e4, 3))  # inspect high-penalty model.

▶ What you'll see: the stronger penalty uses fewer or smaller coefficients among competing features.

In [ ]:
nnz_low_e4 = int(np.sum(np.abs(beta_low_e4) > 1e-8))  # count active weak-penalty features.
nnz_high_e4 = int(np.sum(np.abs(beta_high_e4) > 1e-8))  # count active strong-penalty features.
print("nonzeros weak/strong_e4:", nnz_low_e4, nnz_high_e4)  # inspect sparsity change.
assert nnz_high_e4 <= nnz_low_e4  # verify stronger L1 is not denser here.

▶ What you'll see: stronger L1 selects no more features than weaker L1.

In [ ]:
xpos_e4 = np.arange(3)  # create x positions for grouped bars.
plt.figure(figsize=(5, 3))  # create coefficient comparison plot.
plt.bar(xpos_e4 - 0.18, beta_low_e4, width=0.36, label="weak λ", color="gray")  # plot weak-penalty coefficients.
plt.bar(xpos_e4 + 0.18, beta_high_e4, width=0.36, label="strong λ", color="seagreen")  # plot strong-penalty coefficients.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.xticks(xpos_e4, ["x0", "x1", "x2"])  # label features.
plt.title("Easy 4: correlated-feature selection")  # title the plot.
plt.legend()  # show penalty labels.
plt.show()  # display the chart.

▶ What you'll see: Lasso concentrates weight instead of spreading it freely across correlated predictors.

👀 Takeaway: sparsity can make correlated models simpler, but the chosen representative may change with data noise.

### Easy 5 — Compare Lasso with an unpenalized least-squares fit

**Goal.** Contrast least squares and Lasso on the same design, because Lasso gives up some raw fit to buy smaller and sometimes sparser coefficients. We build it in 4 steps.

In [ ]:
X_e5_raw = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.], [2., 2., 0.]])  # define a small dataset.
y_e5_raw = np.array([1.0, 1.1, 1.9, 2.8, 1.2, 3.0])  # define targets.
X_e5 = standardize(X_e5_raw)  # standardize features for fair penalties.
y_e5 = y_e5_raw - y_e5_raw.mean()  # center target values.
print("data_e5 shape:", X_e5.shape)  # inspect dimensions.

▶ What you'll see: least squares and Lasso will use the same standardized inputs.

In [ ]:
beta_ls_e5 = np.linalg.pinv(X_e5) @ y_e5  # compute a stable unpenalized least-squares solution.
beta_lasso_e5 = lasso_cd(X_e5, y_e5, lam=0.7, steps=200)  # compute a penalized Lasso solution.
print("least squares beta_e5:", np.round(beta_ls_e5, 3))  # inspect unpenalized coefficients.
print("lasso beta_e5:", np.round(beta_lasso_e5, 3))  # inspect penalized coefficients.

▶ What you'll see: Lasso coefficients are shrunk and may include exact zeros.

In [ ]:
sse_ls_e5 = 0.5 * float(np.sum((y_e5 - X_e5 @ beta_ls_e5) ** 2))  # least-squares fit term.
sse_lasso_e5 = 0.5 * float(np.sum((y_e5 - X_e5 @ beta_lasso_e5) ** 2))  # Lasso fit term.
l1_ls_e5 = l1_norm(beta_ls_e5)  # least-squares coefficient size.
l1_lasso_e5 = l1_norm(beta_lasso_e5)  # Lasso coefficient size.
print("fit LS/Lasso_e5:", round(sse_ls_e5, 3), round(sse_lasso_e5, 3))  # inspect fit tradeoff.
print("L1 LS/Lasso_e5:", round(l1_ls_e5, 3), round(l1_lasso_e5, 3))  # inspect size tradeoff.
assert l1_lasso_e5 <= l1_ls_e5 + 1e-8  # Lasso should be no larger in this demo.

▶ What you'll see: Lasso usually has a higher raw fit error but a lower coefficient size.

In [ ]:
plt.figure(figsize=(5, 3))  # create coefficient comparison chart.
plt.plot(beta_ls_e5, marker="o", label="least squares", color="gray")  # plot unpenalized coefficients.
plt.plot(beta_lasso_e5, marker="s", label="Lasso", color="purple")  # plot Lasso coefficients.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.title("Easy 5: fit versus sparse size")  # title the plot.
plt.xlabel("feature index")  # label feature axis.
plt.ylabel("coefficient")  # label coefficient scale.
plt.legend()  # show model labels.
plt.show()  # display the chart.

▶ What you'll see: the penalized solution is pulled toward zero compared with least squares.

👀 Takeaway: Lasso willingly sacrifices some training fit when the smaller coefficient vector is a better full decision.

## 🔴 Advanced

### Advanced 1 — Trace the full coefficient path

**Goal.** Plot every coefficient over a dense λ grid, because Lasso paths reveal when each feature enters or leaves the model. We build it in 4 steps.

In [ ]:
X_a1_raw = np.array([[1., 0., 1., 0.], [0., 1., 1., 1.], [1., 1., 0., 0.], [2., 1., 0., 1.], [0., 2., 1., 1.], [2., 2., 0., 0.]])  # four-feature design.
y_a1_raw = np.array([1.0, 1.2, 2.0, 2.7, 1.4, 3.1])  # targets.
X_a1 = standardize(X_a1_raw)  # standardize features.
y_a1 = y_a1_raw - y_a1_raw.mean()  # center target.
print("advanced path data_a1:", X_a1.shape)  # inspect dimensions.

▶ What you'll see: the path will track four coefficients over many penalties.

In [ ]:
lams_a1 = np.linspace(0.0, 3.0, 16)  # define a dense λ grid.
betas_a1 = np.array([lasso_cd(X_a1, y_a1, lam, steps=220) for lam in lams_a1])  # fit each path point.
print("first beta_a1:", np.round(betas_a1[0], 3))  # inspect nearly unpenalized start.
print("last beta_a1:", np.round(betas_a1[-1], 3))  # inspect heavily penalized end.

▶ What you'll see: the final coefficients are closer to zero than the initial coefficients.

In [ ]:
nnz_a1 = np.sum(np.abs(betas_a1) > 1e-8, axis=1)  # count active features by λ.
print("nonzero counts_a1:", nnz_a1)  # inspect model-size path.
assert nnz_a1[-1] <= nnz_a1[0]  # verify the last model is no denser than the first.

▶ What you'll see: active-feature counts summarize the path as λ increases.

In [ ]:
plt.figure(figsize=(5.4, 3.2))  # create coefficient-path plot.
for j_a1 in range(betas_a1.shape[1]):  # draw one curve per feature.
    plt.plot(lams_a1, betas_a1[:, j_a1], marker="o", markersize=3, label=f"β{j_a1}")  # plot coefficient path.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.title("Advanced 1: Lasso coefficient path")  # title the path plot.
plt.xlabel("λ")  # label penalty axis.
plt.ylabel("coefficient")  # label coefficient scale.
plt.legend()  # show feature labels.
plt.show()  # display the chart.

▶ What you'll see: each feature has its own shrinkage trajectory, and some hit zero earlier than others.

👀 Takeaway: the λ path turns sparsity from a single yes/no result into a visible model-selection curve.

### Advanced 2 — Bootstrap selection stability

**Goal.** Refit Lasso on bootstrap samples and count how often each feature is selected, because sparse feature lists can be unstable when predictors are correlated or data is small. We build it in 5 steps.

In [ ]:
rng_a2 = np.random.default_rng(2)  # create reproducible bootstrap randomness.
n_a2 = 40  # choose sample size.
x0_a2 = rng_a2.normal(size=n_a2)  # true signal feature.
x1_a2 = x0_a2 + 0.2 * rng_a2.normal(size=n_a2)  # correlated competitor.
x2_a2 = rng_a2.normal(size=n_a2)  # noise feature.
X_a2 = standardize(np.column_stack([x0_a2, x1_a2, x2_a2]))  # standardize all features.
y_a2 = 1.5 * x0_a2 + 0.25 * rng_a2.normal(size=n_a2)  # target mostly follows x0.
y_a2 = y_a2 - y_a2.mean()  # center target.
print("corr x0,x1_a2:", round(float(np.corrcoef(X_a2[:, 0], X_a2[:, 1])[0, 1]), 3))  # inspect correlation.

▶ What you'll see: two features are highly correlated, making selection stability worth checking.

In [ ]:
B_a2 = 40  # number of bootstrap refits.
lam_a2 = 1.0  # fixed penalty for the stability experiment.
selected_a2 = np.zeros((B_a2, X_a2.shape[1]), dtype=bool)  # store selected-feature masks.
print("bootstrap runs_a2:", B_a2, "lambda_a2:", lam_a2)  # inspect experiment settings.

▶ What you'll see: the experiment will refit the same λ many times on resampled rows.

In [ ]:
for b_a2 in range(B_a2):  # repeat bootstrap resampling.
    idx_a2 = rng_a2.integers(0, n_a2, size=n_a2)  # sample rows with replacement.
    beta_a2 = lasso_cd(X_a2[idx_a2], y_a2[idx_a2], lam_a2, steps=180)  # fit Lasso on bootstrap sample.
    selected_a2[b_a2] = np.abs(beta_a2) > 1e-8  # record which features survived.
freq_a2 = selected_a2.mean(axis=0)  # compute selection frequencies.
print("selection frequencies_a2:", np.round(freq_a2, 3))  # inspect stability of each feature.

▶ What you'll see: correlated signal features may split selection frequency instead of one feature winning every time.

In [ ]:
assert np.all((freq_a2 >= 0) & (freq_a2 <= 1))  # verify frequencies are probabilities.
most_stable_a2 = int(np.argmax(freq_a2))  # identify most frequently selected feature.
print("most selected feature_a2:", most_stable_a2)  # inspect the stability winner.

▶ What you'll see: the most frequent feature is the one Lasso trusted most often under resampling.

In [ ]:
plt.figure(figsize=(4.5, 3))  # create stability plot.
plt.bar(["x0", "x1", "x2"], freq_a2, color="darkorange")  # show selection probability per feature.
plt.ylim(0, 1.05)  # keep probability axis bounded.
plt.title("Advanced 2: bootstrap selection stability")  # title the plot.
plt.ylabel("selection frequency")  # label y-axis.
plt.show()  # display the chart.

▶ What you'll see: stable features have bars near 1, unstable or irrelevant features have lower bars.

👀 Takeaway: sparsity is useful, but selected feature identities should be checked for stability when features overlap.

### Advanced 3 — Demonstrate the missing scale pitfall

**Goal.** Fit Lasso before and after standardization, because an L1 penalty is unfair when feature scales differ. We build it in 5 steps.

In [ ]:
x0_a3 = np.linspace(0, 10, 30)  # feature on a large scale.
x1_a3 = x0_a3 / 10.0  # same information on a smaller scale.
X_raw_a3 = np.column_stack([x0_a3, x1_a3])  # combine differently scaled copies.
y_a3 = 2.0 * x1_a3 + 0.05 * np.sin(x0_a3)  # target written naturally on the small-scale feature.
y_a3 = y_a3 - y_a3.mean()  # center target.
print("std raw features_a3:", np.round(X_raw_a3.std(axis=0), 3))  # inspect scale mismatch.

▶ What you'll see: the two columns carry similar information but have very different standard deviations.

In [ ]:
X_center_a3 = X_raw_a3 - X_raw_a3.mean(axis=0)  # center without scaling.
X_std_a3 = standardize(X_raw_a3)  # center and scale.
lam_a3 = 0.8  # fixed penalty used for both fits.
print("lambda_a3:", lam_a3)  # inspect common penalty strength.

▶ What you'll see: the same λ will be applied to unfair raw columns and fair standardized columns.

In [ ]:
beta_raw_a3 = lasso_cd(X_center_a3, y_a3, lam_a3, steps=250)  # fit without scaling.
beta_std_a3 = lasso_cd(X_std_a3, y_a3, lam_a3, steps=250)  # fit with standardization.
print("raw-scale beta_a3:", np.round(beta_raw_a3, 3))  # inspect scale-biased fit.
print("standardized beta_a3:", np.round(beta_std_a3, 3))  # inspect fair fit.

▶ What you'll see: coefficient magnitudes are not directly comparable until features are standardized.

In [ ]:
pred_raw_a3 = X_center_a3 @ beta_raw_a3  # raw-scale predictions.
pred_std_a3 = X_std_a3 @ beta_std_a3  # standardized predictions.
rmse_raw_a3 = float(np.sqrt(np.mean((y_a3 - pred_raw_a3) ** 2)))  # raw-scale error.
rmse_std_a3 = float(np.sqrt(np.mean((y_a3 - pred_std_a3) ** 2)))  # standardized error.
print("RMSE raw/std_a3:", round(rmse_raw_a3, 3), round(rmse_std_a3, 3))  # inspect fit difference.
assert rmse_raw_a3 >= 0 and rmse_std_a3 >= 0  # verify valid errors.

▶ What you'll see: both fits are valid, but the raw-scale coefficients reflect units as much as signal.

In [ ]:
xpos_a3 = np.arange(2)  # positions for grouped bars.
plt.figure(figsize=(5, 3))  # create scale comparison plot.
plt.bar(xpos_a3 - 0.18, beta_raw_a3, width=0.36, label="raw scale", color="crimson")  # plot raw-scale coefficients.
plt.bar(xpos_a3 + 0.18, beta_std_a3, width=0.36, label="standardized", color="seagreen")  # plot standardized coefficients.
plt.axhline(0, color="black", linewidth=0.8)  # mark zero.
plt.xticks(xpos_a3, ["feature 0", "feature 1"])  # label features.
plt.title("Advanced 3: scale affects L1 selection")  # title the plot.
plt.legend()  # show labels.
plt.show()  # display the chart.

▶ What you'll see: the L1 penalty behaves differently when feature units differ.

👀 Takeaway: standardize features before Lasso unless the original units are intentionally part of the penalty design.

### Advanced 4 — Compare raw fit, penalty, and full score across λ

**Goal.** Decompose the objective across λ values, because a lower raw fit error can lose after the L1 cost is included. We build it in 5 steps.

In [ ]:
X_a4_raw = np.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 1., 0.], [0., 2., 1.], [2., 2., 0.]])  # define data.
y_a4_raw = np.array([1.0, 1.1, 1.9, 2.8, 1.2, 3.0])  # define targets.
X_a4 = standardize(X_a4_raw)  # standardize features.
y_a4 = y_a4_raw - y_a4_raw.mean()  # center target.
lams_a4 = np.array([0.0, 0.3, 0.8, 1.5])  # candidate penalties.
print("candidate lambdas_a4:", lams_a4)  # inspect grid.

▶ What you'll see: each λ will produce a fit term, penalty term, and total score.

In [ ]:
betas_a4 = np.array([lasso_cd(X_a4, y_a4, lam, steps=220) for lam in lams_a4])  # fit all candidates.
fit_terms_a4 = np.array([0.5 * np.sum((y_a4 - X_a4 @ b_a4) ** 2) for b_a4 in betas_a4])  # raw fit terms.
l1_terms_a4 = np.array([l1_norm(b_a4) for b_a4 in betas_a4])  # coefficient sizes.
print("fit terms_a4:", np.round(fit_terms_a4, 3))  # inspect raw fit costs.
print("L1 sizes_a4:", np.round(l1_terms_a4, 3))  # inspect complexity costs before λ scaling.

▶ What you'll see: raw fit tends to worsen as λ grows, while coefficient size tends to shrink.

In [ ]:
full_scores_a4 = fit_terms_a4 + lams_a4 * l1_terms_a4  # compute each λ's own objective score.
best_idx_a4 = int(np.argmin(full_scores_a4))  # choose by full objective.
print("full scores_a4:", np.round(full_scores_a4, 3))  # inspect comparable objective values.
print("best lambda by full score_a4:", float(lams_a4[best_idx_a4]))  # inspect winner.
assert best_idx_a4 >= 0  # sanity check selected index.

▶ What you'll see: the selected λ is based on fit plus penalty, not raw fit alone.

In [ ]:
nnz_a4 = np.sum(np.abs(betas_a4) > 1e-8, axis=1)  # count nonzero coefficients.
print("nonzeros_a4:", nnz_a4)  # inspect model size by λ.
assert nnz_a4[-1] <= nnz_a4[0]  # stronger penalty is no denser in this sweep.

▶ What you'll see: stronger penalties use no more active coefficients than the no-penalty fit here.

In [ ]:
plt.figure(figsize=(5.4, 3))  # create objective decomposition plot.
plt.plot(lams_a4, fit_terms_a4, marker="o", label="raw fit")  # plot fit term.
plt.plot(lams_a4, lams_a4 * l1_terms_a4, marker="s", label="λL1")  # plot scaled penalty.
plt.plot(lams_a4, full_scores_a4, marker="^", label="full score")  # plot total objective.
plt.axvline(lams_a4[best_idx_a4], color="red", linestyle="--", label="best full score")  # mark winner.
plt.title("Advanced 4: compare complete scores")  # title the plot.
plt.xlabel("λ")  # label penalty axis.
plt.ylabel("value")  # label objective value.
plt.legend()  # show component labels.
plt.show()  # display the chart.

▶ What you'll see: the raw fit curve and full-score curve answer different questions.

👀 Takeaway: forgetting the cost term changes the selection mechanism and can pick the wrong model.

### Advanced 5 — Build a validation-first Lasso selection workflow

**Goal.** Combine a λ sweep, validation RMSE, sparsity, and the lesson's full-score thinking, because durable model choice needs both predictive evidence and model cost. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(5)  # create reproducible data randomness.
X_all_a5 = rng_a5.normal(size=(60, 6))  # create six candidate features.
true_beta_a5 = np.array([1.6, 0.0, -1.2, 0.0, 0.0, 0.7])  # define a sparse ground-truth pattern.
y_all_a5 = X_all_a5 @ true_beta_a5 + 0.35 * rng_a5.normal(size=60)  # generate noisy targets.
X_all_a5 = standardize(X_all_a5)  # standardize feature scales.
y_all_a5 = y_all_a5 - y_all_a5.mean()  # center target.
print("true nonzeros_a5:", np.where(true_beta_a5 != 0)[0])  # inspect hidden sparse signal.

▶ What you'll see: only three of six features truly drive the synthetic target.

In [ ]:
train_a5 = np.arange(0, 40)  # first 40 rows for training.
val_a5 = np.arange(40, 60)  # last 20 rows for validation.
X_train_a5, y_train_a5 = X_all_a5[train_a5], y_all_a5[train_a5]  # training split.
X_val_a5, y_val_a5 = X_all_a5[val_a5], y_all_a5[val_a5]  # validation split.
lams_a5 = np.array([0.0, 0.2, 0.6, 1.0, 1.8, 3.0])  # candidate λ values.
print("train/val sizes_a5:", len(train_a5), len(val_a5))  # inspect split sizes.

▶ What you'll see: the λ decision will be made on rows not used to fit β.

In [ ]:
betas_a5 = np.array([lasso_cd(X_train_a5, y_train_a5, lam, steps=250) for lam in lams_a5])  # fit all candidates.
val_rmse_a5 = np.array([np.sqrt(np.mean((y_val_a5 - X_val_a5 @ b_a5) ** 2)) for b_a5 in betas_a5])  # validation RMSE.
nonzeros_a5 = np.sum(np.abs(betas_a5) > 1e-8, axis=1)  # model sizes.
print("validation RMSE_a5:", np.round(val_rmse_a5, 3))  # inspect future-fit proxy.
print("nonzeros_a5:", nonzeros_a5)  # inspect sparsity by λ.

▶ What you'll see: different λ values trade validation error against active feature count.

In [ ]:
complexity_cost_a5 = 0.03 * nonzeros_a5  # define a small explicit cost per active feature for decision-making.
decision_a5 = val_rmse_a5 + complexity_cost_a5  # combine predictive error and operational complexity.
best_idx_a5 = int(np.argmin(decision_a5))  # choose the lowest full decision score.
print("decision scores_a5:", np.round(decision_a5, 3))  # inspect comparable full scores.
print("best λ_a5:", float(lams_a5[best_idx_a5]), "active features:", int(nonzeros_a5[best_idx_a5]))  # inspect selected model.
assert decision_a5[best_idx_a5] == np.min(decision_a5)  # verify selected index.

▶ What you'll see: the winner balances validation performance with a simple sparsity cost.

In [ ]:
plt.figure(figsize=(5.4, 3))  # create workflow summary plot.
plt.plot(lams_a5, val_rmse_a5, marker="o", label="validation RMSE")  # plot predictive error.
plt.plot(lams_a5, decision_a5, marker="s", label="RMSE + sparsity cost")  # plot full decision score.
plt.axvline(lams_a5[best_idx_a5], color="red", linestyle="--", label="selected λ")  # mark selected λ.
plt.title("Advanced 5: validation-first Lasso selection")  # title the plot.
plt.xlabel("λ")  # label penalty axis.
plt.ylabel("score")  # label score scale.
plt.legend()  # show curve labels.
plt.show()  # display the chart.

▶ What you'll see: the chosen λ is the lowest full decision score, not automatically the densest or sparsest model.

👀 Takeaway: a strong Lasso workflow compares validation behavior and sparsity cost on one explicit decision scale.